<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Topic_Karhutla%2BErupsi_top_author_sentimen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Install Dependency

In [ ]:
!pip install -q -U google-genai pandas tqdm emoji

## 1. Import Library

In [ ]:
import re
import json
import time
import hashlib
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display
from google import genai
from google.genai import types

try:
    import emoji
    HAS_EMOJI_LIB = True
except ImportError:
    HAS_EMOJI_LIB = False
    print("[WARN] library 'emoji' tidak ada -> emoji akan dibuang, bukan dikonversi jadi teks.")

import anthropic

print("Library siap.")


ModuleNotFoundError: No module named 'anthropic'

## 2. Set API Key Anthropic

Key diketik lewat input tersembunyi (tidak ke-log di notebook), jadi aman
kalau notebook ini nanti di-upload ke GitHub.

Alternatif lebih nyaman: pakai fitur **Secrets** Colab (ikon kunci di sidebar
kiri), simpan sebagai `ANTHROPIC_API_KEY`, lalu ganti isi cell ini dengan:

```python
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
```


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

import anthropic
client = anthropic.Anthropic()

print("API key sudah di-set.")

## 3. Konfigurasi

Edit bagian ini kalau nama kolom, daftar tema, atau bobot ranking berubah.
Semua cell di bawah memakai variabel dari sini.


In [12]:
# ---------------------------------------------------------------
# Link Google Sheets (harus "anyone with link can view")
# ---------------------------------------------------------------
# ---------------------------------------------------------------
# Step 1 - Load Data dari file .xlsx di Google Drive
# ---------------------------------------------------------------
!pip install -q gdown openpyxl

import re
import gdown
import pandas as pd

DRIVE_XLSX_URL = "https://docs.google.com/spreadsheets/d/1RWje0fuPtl1jHVzibr480gWfK0ivJkbY/edit?usp=sharing&ouid=116825097454650626545&rtpof=true&sd=true"

# Nama sheet di dalam file xlsx -> GANTI sesuai nama tab aslinya kalau beda
SHEET_NAME = "Sheet1"


def extract_drive_file_id(url: str) -> str:
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    match = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    raise ValueError("Tidak bisa menemukan file ID dari URL. Pastikan format link Google Drive benar.")


file_id = extract_drive_file_id(DRIVE_XLSX_URL)
local_path = "data_input.xlsx"

print(f"[INFO] File ID terdeteksi: {file_id}")
print("Mengunduh file .xlsx dari Google Drive...")
gdown.download(f"https://drive.google.com/uc?id={file_id}", local_path, quiet=False)

# cek dulu nama-nama sheet yang ada di file, buat mastiin SHEET_NAME di atas benar
xls = pd.ExcelFile(local_path)
print(f"\n[INFO] Sheet yang tersedia di file ini: {xls.sheet_names}")

df_raw = pd.read_excel(local_path, sheet_name=SHEET_NAME, skiprows=1)  # baris 1 = judul, baris 2 = header kolom

print(f"\n[OK] {len(df_raw)} baris, {len(df_raw.columns)} kolom berhasil dimuat")
print("Kolom:", list(df_raw.columns))
display(df_raw.head(5))

# ---------------------------------------------------------------
# Daftar tema (polusi_udara sudah digabung ke karhutla)
# ---------------------------------------------------------------
THEMES = ["karhutla", "erupsi", "lainnya"]
THEME_DESCRIPTIONS = {
    "karhutla": (
        "karhutla, kebakaran hutan, kebakaran lahan, kebakaran hutan dan lahan, titik api, hotspot, "
        "lahan gambut, gambut terbakar, water bombing, manggala agni, hujan buatan, TMC, "
        "teknologi modifikasi cuaca, langit kuning, darurat asap, kabut asap, asap kebakaran, "
        "jarak pandang terbatas akibat asap, ISPU, kualitas udara, indeks standar pencemar udara, "
        "polusi udara, udara tidak sehat, udara berbahaya, PM2.5, PM10, partikulat, emisi kendaraan, "
        "emisi industri, emisi karbon, karhutla Riau, karhutla Kalimantan, karhutla Sumatera, "
        "BNPB kebakaran, damkar hutan, pemadaman hutan, api hutan, bara api, sekat kanal, "
        "restorasi gambut, ISPA akibat asap, sekolah diliburkan akibat asap, penerbangan terganggu asap, "
        "kebakaran lahan gambut, asap lintas batas, ekspor asap, kanal bersekat, korporasi pembakar lahan, "
        "karhutla musim kemarau, El Nino kebakaran, kebakaran hutan Sumatera Selatan, kebakaran hutan Jambi, "
        "kebakaran hutan Kalimantan Tengah, kebakaran hutan Kalimantan Barat, kebakaran hutan Kalimantan Timur, "
        "hutan lindung terbakar, cuaca panas ekstrem picu kebakaran, kekeringan picu karhutla, "
        "operasi udara pemadaman, helikopter water bombing, pesawat pemadam kebakaran, BPBD kebakaran, "
        "KLHK segel lahan, perusahaan sawit karhutla, moratorium sawit, denda karhutla, gugatan karhutla, "
        "izin HGU dicabut karhutla, hotspot satelit, citra satelit hotspot, membakar lahan, bakar lahan, "
        "buka lahan dengan cara dibakar, pembakaran lahan sengaja, asap tebal, asap pekat, polusi asap, "
        "ISPU tidak sehat, waspada karhutla, siaga darurat karhutla, status darurat asap, korban asap, "
        "rumah sakit ISPA, penerbangan dibatalkan akibat asap, bandara ditutup akibat asap, "
        "jarak pandang di bawah satu kilometer, kabut asap pekat, kualitas udara memburuk"
    ),
    "erupsi": (
        "erupsi, erupsi gunung berapi, letusan gunung, letusan, gunung meletus, gunung berapi aktif, "
        "abu vulkanik, hujan abu, awan panas guguran, APG, wedhus gembel, lahar dingin, lahar panas, "
        "lahar hujan, banjir lahar, status awas, status siaga, status waspada, status normal, PVMBG, "
        "Badan Geologi, evakuasi warga, pengungsian, zona merah, radius bahaya, radius aman, "
        "magma, kawah, dapur magma, guguran lava, lava pijar, sinar api, dentuman, suara gemuruh, "
        "gempa vulkanik, tremor vulkanik, aktivitas vulkanik meningkat, bandara ditutup abu vulkanik, "
        "penerbangan terdampak abu vulkanik, Gunung Semeru, Gunung Merapi, Gunung Ibu, Gunung Lewotobi, "
        "Gunung Anak Krakatau, Gunung Kerinci, Gunung Marapi, Gunung Dukono, Gunung Karangetang, "
        "Gunung Ile Lewotolok, Gunung Sinabung, Gunung Agung, Gunung Raung, Gunung Rinjani, Gunung Ijen, "
        "Gunung Slamet, Gunung Bromo, Gunung Kelud, Gunung Tangkuban Perahu, Gunung Papandayan, "
        "Gunung Ciremai, Gunung Gede, Gunung Salak, Gunung Talang, Gunung Soputan, Gunung Lokon, "
        "Gunung Awu, Gunung Gamalama, Gunung Ruang, Gunung Iya, Gunung Egon, Gunung Rokatenda, "
        "Gunung Kelimutu, Gunung Sirung, gunung berstatus awas, gunung meletus dahsyat, semburan lava, "
        "guguran awan panas, kolom abu, tinggi kolom letusan, letusan freatik, letusan eksplosif, "
        "aktivitas seismik gunung, PVMBG naikkan status, PVMBG turunkan status, zona bahaya erupsi, "
        "warga mengungsi akibat erupsi, posko pengungsian erupsi, jalur evakuasi erupsi, "
        "sirine peringatan dini erupsi, rekomendasi PVMBG, Badan Geologi ESDM, "
        "atap rumah roboh akibat abu vulkanik, petani gagal panen akibat abu vulkanik, "
        "penerbangan dialihkan akibat abu vulkanik, notam bandara abu vulkanik, VONA aviation, "
        "kode warna penerbangan gunung berapi"
    ),
    "lainnya": "topik di luar dua kategori di atas",
}

# ---------------------------------------------------------------
# Whitelist & keyword akun media (boleh ditambah)
# nama akun disimpan dalam bentuk sudah dinormalisasi: huruf kecil, tanpa "@", tanpa simbol/angka
# ---------------------------------------------------------------
MEDIA_ACCOUNTS = {
    "detikcom", "kompascom", "cnnindonesia", "tempodotco", "antaranews",
    "cnbcindonesia", "liputan6", "liputan6dotcom", "sctv", "tvonenews", "kumparan",
    "beritasatu", "metrotvnews", "tribunnews", "republikaonline", "suaradotcom",
    "jawapos", "bbcindonesia", "voaindonesia", "narasitv", "sindonews",
    "mediaindonesia", "radioelshinta", "pikiranrakyat",
}
MEDIA_KEYWORDS = [
    "news", "media", "tv", "radio", "koran", "pers", "redaksi",
    "newsroom", "resmi", "official", "humas", "pemerintah", "pemkot", "pemprov",
    "tribun", "portal", "berita", "radar",
]

# ---------------------------------------------------------------
# Model & parameter LLM
# ---------------------------------------------------------------
from google import genai
from google.genai import types

GEMINI_MODEL = "gemini-3.1-flash-lite"
gemini_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

SLEEP_BETWEEN_CALLS = 4

# Bobot skor ranking top author (harus berjumlah 1.0)
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15

# ---------------------------------------------------------------
# Parameter batch & ranking
# ---------------------------------------------------------------
BATCH_SIZE_CLASSIFY = 10           # jumlah teks per batch saat klasifikasi tema
BATCH_SIZE_NER = 20                # jumlah teks per batch saat NER
MAX_WORKERS_NER = 4                # jumlah panggilan API NER yang jalan bersamaan (paralel)
TOP_N_AUTHORS_PER_THEME = 10       # jumlah top author yang diambil per tema
MAX_POSTS_PER_AUTHOR_SUMMARY = 15  # maks jumlah post per author yang dikirim ke prompt summarization


def build_theme_prompt(batch_texts):
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    themes_desc = "\n".join(f"- {k}: {v}" for k, v in THEME_DESCRIPTIONS.items())
    return f"""Klasifikasikan tema dari tiap teks berbahasa Indonesia berikut ke SATU dari kategori berikut:
{themes_desc}

PENTING: baca dan pertimbangkan SEMUA kategori dengan bobot yang sama sebelum memutuskan.
Jangan asumsikan satu tema lebih mungkin daripada tema lain.

Teks:
{numbered}

Jawab HANYA dengan JSON array, tanpa penjelasan, tanpa markdown code block. Contoh format
(nilai "theme" di contoh ini HANYA ilustrasi format, bukan indikasi jawaban yang benar):
[
  {{"index": 1, "theme": "erupsi", "confidence": 0.9}},
  {{"index": 2, "theme": "karhutla", "confidence": 0.8}},
  {{"index": 3, "theme": "lainnya", "confidence": 0.6}}
]
Jumlah item HARUS sama persis dengan jumlah teks ({len(batch_texts)} item)."""


print("Konfigurasi siap.")

[INFO] File ID terdeteksi: 1RWje0fuPtl1jHVzibr480gWfK0ivJkbY
Mengunduh file .xlsx dari Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1RWje0fuPtl1jHVzibr480gWfK0ivJkbY
To: /content/data_input.xlsx
100%|██████████| 11.6M/11.6M [00:00<00:00, 170MB/s]



[INFO] Sheet yang tersedia di file ini: ['Sheet1']

[OK] 66406 baris, 13 kolom berhasil dimuat
Kolom: ['No', 'Type', 'Headline', 'Mentions', 'Date', 'Link', 'Media', 'Sentiment', 'Author', 'Followers', 'Retweeted', 'Favourited', 'Location']


,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited,Location
0,1,mention,NaN,Prabowo: Karhutla Jangan Sampai ke Zona Inti I...,2026-09-07 17:40:17,https://twitter.com/web/statuses/2096911425180...,Twitter,Positive,@kompascom,8042391,0,0,Jakarta
1,2,mention,NaN,Presiden Prabowo Subianto memberikan instruksi...,2026-09-07 17:33:56,https://twitter.com/web/statuses/2096909830162...,Twitter,Positive,@makcrigis_,177,0,0,NaN
2,3,mention,Prabowo Tegas Larang Segala Bentuk Pembakaran ...,Jakarta: Presiden RI Prabowo Subianto menginst...,2026-09-07 17:33:21,https://www.metrotvnews.com/read/b2lC62aV-prab...,News,Neutral,www.metrotvnews.com,0,0,0,NaN
3,4,mention,Prabowo Tegas Larang Segala Bentuk Pembakaran ...,Jakarta: Presiden RI Prabowo Subianto menginst...,2026-09-07 17:33:21,https://www.metrotvnews.com/read/b2lC62aV-prab...,News,Positive,www.metrotvnews.com,0,0,0,NaN
4,5,mention,NaN,Karhutla Kepulauan Meranti Tembus 133 Hektare ...,2026-09-07 17:33:20,https://twitter.com/web/statuses/2096909679599...,Twitter,Positive,@bukamata18,156,0,0,Jakarta


Konfigurasi siap.


In [ ]:
# ---------------------------------------------------------------
# Fungsi bantu untuk klasifikasi tema (keyword + LLM few-shot, multi-label)
# ---------------------------------------------------------------

def keyword_match_theme(text):
    """Prefilter cepat berdasarkan keyword di THEME_DESCRIPTIONS, TIDAK dikirim ke LLM.
    Return SATU tema (single) atau None kalau tidak ketemu -> lanjut ke LLM."""
    text_lower = str(text).lower()
    for theme in THEMES:
        if theme == "lainnya":
            continue
        desc = THEME_DESCRIPTIONS[theme]
        kw_list = [re.escape(k.strip()) for k in desc.split(",") if k.strip()]
        pattern = "|".join(kw_list)
        if re.search(pattern, text_lower):
            return theme
    return None  # tidak ketemu keyword apapun, biar LLM yang putuskan


THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in THEMES)

FEWSHOT_THEME_EXAMPLES = """
Contoh klasifikasi (belajar dari pola ini):

Teks: "Kebakaran hutan di Riau menyebabkan asap tebal, sekolah diliburkan karena kualitas udara memburuk"
Jawaban: {"themes": ["karhutla", "polusi_udara"], "primary_theme": "karhutla", "confidence": 0.95}

Teks: "ISPU Jakarta hari ini masuk kategori tidak sehat akibat kepadatan kendaraan dan emisi industri"
Jawaban: {"themes": ["polusi_udara"], "primary_theme": "polusi_udara", "confidence": 0.9}

Teks: "Gunung Anak Krakatau erupsi, warga diminta waspada terhadap sebaran abu vulkanik"
Jawaban: {"themes": ["erupsi"], "primary_theme": "erupsi", "confidence": 0.95}

Teks: "Erupsi gunung memicu hujan abu yang turut memperburuk kualitas udara di wilayah sekitar"
Jawaban: {"themes": ["erupsi", "polusi_udara"], "primary_theme": "erupsi", "confidence": 0.85}

Teks: "Harga cabai naik jelang lebaran, pedagang mengeluh"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.95}
"""


def build_theme_prompt(batch_texts):
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    return f"""Kamu adalah classifier tema untuk cuitan/berita berbahasa Indonesia.

Daftar tema yang tersedia:
{THEME_LIST_STR}

{FEWSHOT_THEME_EXAMPLES}

Sekarang klasifikasikan SETIAP teks di bawah ini. SATU teks BOLEH punya lebih dari
1 tema kalau memang relevan (lihat contoh ke-1 dan ke-4 di atas), tapi tetap
tentukan "primary_theme" sebagai tema yang paling dominan/utama dibahas.
Jika tidak cocok ke tema manapun selain "lainnya", gunakan "lainnya" saja.

Teks:
{numbered}

Jawab HANYA dengan JSON array (tanpa penjelasan, tanpa markdown code block), formatnya:
[
  {{"index": 1, "themes": ["karhutla", "polusi_udara"], "primary_theme": "karhutla", "confidence": 0.9}},
  {{"index": 2, "themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.6}}
]
Jumlah item HARUS sama persis dengan jumlah teks di atas ({len(batch_texts)} item)."""


def classify_theme_batch(batch_texts, retry=6):
    prompt = build_theme_prompt(batch_texts)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if len(parsed) != len(batch_texts):
                raise ValueError(f"Jumlah hasil ({len(parsed)}) != jumlah input ({len(batch_texts)})")
            return parsed
        except Exception as e:
            is_overload = "503" in str(e) or "UNAVAILABLE" in str(e)
            wait = 20 * (attempt + 1) if is_overload else 5 * (attempt + 1)
            print(f"[WARN] Klasifikasi tema batch gagal (percobaan {attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    print(f"[ERROR] Batch ini gagal total setelah {retry} percobaan, dicap 'lainnya' sementara")
    return [
        {"index": i + 1, "themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.0}
        for i in range(len(batch_texts))
    ]


print("Fungsi klasifikasi tema siap (keyword prefilter + LLM few-shot multi-label).")

## 4. Step 1 — Load Data dari Google Sheets

Output: tabel mentah + daftar kolom, buat konfirmasi data ke-load dengan benar.


In [ ]:
def build_csv_url(sheet_url: str) -> str:
    sheet_id = re.search(r"/d/([a-zA-Z0-9-_]+)", sheet_url).group(1)
    gid_match = re.search(r"gid=(\d+)", sheet_url)
    gid = gid_match.group(1) if gid_match else "0"
    return f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

csv_url = build_csv_url(SHEET_URL)
df_raw = pd.read_csv(csv_url, skiprows=1)  # baris 1 = judul, baris 2 = header kolom

print(f"[OK] {len(df_raw)} baris, {len(df_raw.columns)} kolom berhasil dimuat")
print("Kolom:", list(df_raw.columns))
display(df_raw.head(5))


In [ ]:
# ---------------------------------------------------------------
# Step 1b - Deteksi & buang akun media, dijalankan SEBELUM cleaning teks
# supaya proses cleaning tidak buang waktu di baris yang toh dibuang juga
# ---------------------------------------------------------------
import re

MEDIA_TYPE_ALWAYS = {"news"}
DOMAIN_SUFFIXES = (".com", ".id", ".co", ".net", ".org")
DOMAIN_TEXT_SUFFIXES = ("dotcom", "dotid", "dotco", "dotnet", "dotorg")  # banyak outlet nulis domain sbg teks

MEDIA_ACCOUNTS = {
    # nasional
    "detikcom", "kompascom", "cnnindonesia", "tempodotco", "antaranews",
    "cnbcindonesia", "liputan6dotcom", "liputan6", "sctv", "tvonenews", "kumparan",
    "beritasatu", "metrotvnews", "tribunnews", "republikaonline", "suaradotcom",
    "jawapos", "bbcindonesia", "voaindonesia", "narasitv", "sindonews",
    "vivacoid", "bisniscom", "hariankompas", "tirtoid", "alineadotid",
    "inilahcom", "merdekadotcom", "okezone", "idntimes", "medcomid",
    "rri", "tvri", "jpnndotcom", "grid", "inewsdotid", "fajar",
    "pikiranrakyat", "gatra", "nuonline",
    # malaysia (banyak muncul di data ini)
    "awani", "bharianmy", "bernamadotcom", "utusandotcom", "sinarharian",
    "malaysiakini", "thestar", "nst", "theedgemarkets",
}

# kata kunci PANJANG & SPESIFIK -> aman dicek sebagai substring (jarang salah tangkap)
MEDIA_KEYWORDS_SUBSTRING = [
    "news", "media", "redaksi", "newsroom", "official", "humas", "koran",
    "awani", "gazette",
    # NOTE: keyword institusi pemerintah (kemen, bnpb, bmkg, bpbd, polri, setneg)
    # SENGAJA tidak dimasukkan -> akun kementerian/lembaga tetap dihitung sbg
    # kandidat top author, tidak dianggap "media".
]
# kata kunci PENDEK/AMBIGU -> harus exact-match per kata (hindari "persija" ke-tangkep "pers")
MEDIA_KEYWORDS_EXACT = ["tv", "pers", "radio"]


def normalize_account(text: str) -> str:
    """Huruf kecil + angka saja, buat cocokin ke whitelist/domain suffix.
    PENTING: jangan buang digit -> banyak nama outlet pakai angka, mis. 'liputan6dotcom'."""
    return re.sub(r"[^a-z0-9]", "", str(text).strip().lower())


def contains_media_keyword(author_lower: str) -> bool:
    # cek substring dulu (keyword yang aman/spesifik)
    if any(kw in author_lower for kw in MEDIA_KEYWORDS_SUBSTRING):
        return True
    # baru cek exact-token buat keyword yang rawan ambigu
    tokens = re.split(r"[^a-z]+", author_lower)
    return any(tok in MEDIA_KEYWORDS_EXACT for tok in tokens if tok)


def is_media_account(row) -> bool:
    media_val = str(row.get(COL_MEDIA, "")).strip().lower()
    author_raw = str(row.get(COL_AUTHOR_ID, "")).strip().lower().lstrip("@")
    author_norm = normalize_account(author_raw)

    if media_val in MEDIA_TYPE_ALWAYS:
        return True
    if author_norm in MEDIA_ACCOUNTS:
        return True
    if contains_media_keyword(author_raw):
        return True
    if author_raw.endswith(DOMAIN_SUFFIXES):
        return True
    if author_norm.endswith(DOMAIN_TEXT_SUFFIXES):
        return True
    return False


# pastikan Followers numeric dulu, biar sort_values di QC akurat
df_raw[COL_FOLLOWERS] = pd.to_numeric(df_raw.get(COL_FOLLOWERS, 0), errors="coerce").fillna(0)

# simpan salinan data mentah utuh dulu, jaga-jaga kalau nanti butuh cek ulang baris media
df_raw_original = df_raw.copy()

df_raw["is_media"] = df_raw.apply(is_media_account, axis=1)

n_media = df_raw["is_media"].sum()
n_nonmedia = (~df_raw["is_media"]).sum()
print(f"[INFO] {n_media} baris media (dibuang SEBELUM cleaning)")
print(f"[INFO] {n_nonmedia} baris non-media (lanjut ke cleaning)")

print("\nContoh deteksi per akun unik:")
display(
    df_raw[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .head(20)
)

print("\n[QC 1] Followers tertinggi yang KE-FLAG MEDIA — cek jangan sampai personal/influencer besar salah kena filter:")
display(
    df_raw[df_raw["is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .sort_values(COL_FOLLOWERS, ascending=False)
    [[COL_AUTHOR_ID, COL_MEDIA, COL_FOLLOWERS]]
    .head(15)
)

print("\n[QC 2] Followers tertinggi yang TIDAK ke-flag — cek jangan sampai media lolos heuristik:")
display(
    df_raw[~df_raw["is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .sort_values(COL_FOLLOWERS, ascending=False)
    [[COL_AUTHOR_ID, COL_MEDIA, COL_FOLLOWERS]]
    .head(15)
)

df_raw = df_raw[~df_raw["is_media"]].drop(columns=["is_media"]).reset_index(drop=True)
print(f"\n[INFO] df_raw sekarang berisi {len(df_raw)} baris non-media, siap dibersihkan")

## 5. Step 2 — Cleaning Teks

Menggabungkan `Headline` + `Mentions`, lalu membersihkan: URL, mention,
elongasi huruf ("parahhhh" -> "parah"), emoji -> teks, normalisasi kata alay,
dan dedup post identik/retweet.

### 5a. Fungsi cleaning


In [ ]:
KAMUS_ALAY = {
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "tdk": "tidak", "gk": "tidak",
    "kaga": "tidak", "kagak": "tidak", "ngga": "tidak", "enggak": "tidak", "tak": "tidak",
    "gaada": "tidak ada", "bgt": "banget", "bnget": "banget", "bgttt": "banget",
    "bgt2": "banget", "banget2": "banget", "sangattt": "sangat",
    "yg": "yang", "yng": "yang", "krn": "karena", "krna": "karena", "karna": "karena",
    "sm": "sama", "sm2": "sama-sama", "sama2": "sama-sama",
    "utk": "untuk", "u/": "untuk", "bwt": "buat", "buatt": "buat", "utuk": "untuk",
    "dr": "dari", "drpd": "daripada",
    "skrg": "sekarang", "skrng": "sekarang", "skarang": "sekarang", "skg": "sekarang",
    "dgn": "dengan", "dg": "dengan", "dngan": "dengan",
    "org": "orang", "orng": "orang",
    "tp": "tapi", "tpi": "tapi", "spt": "seperti", "kayak": "seperti", "kaya": "seperti", "kyk": "seperti",
    "blm": "belum", "jgn": "jangan", "jgnkan": "jangankan", "jgnlah": "janganlah",
    "emg": "memang", "emang": "memang", "emank": "memang", "emgnya": "memangnya", "emangnya": "memangnya",
    "jd": "jadi", "jadinya": "jadinya", "jg": "juga", "aja": "saja", "aj": "saja",
    "udah": "sudah", "udh": "sudah", "dah": "sudah", "sdh": "sudah",
    "gmn": "bagaimana", "gmna": "bagaimana", "gmana": "bagaimana", "gimana": "bagaimana",
    "knp": "kenapa", "knpa": "kenapa", "napa": "kenapa", "ngapain": "sedang apa", "ngapa": "kenapa",
    "kalo": "kalau", "klo": "kalau",
    "trs": "terus", "trus": "terus", "gini": "begini", "gitu": "begitu", "gt": "begitu", "gtu": "begitu",
    "sy": "saya", "km": "kamu", "gw": "saya", "gwa": "saya", "gua": "saya", "gue": "saya",
    "ane": "saya", "aq": "saya", "aku": "saya",
    "lo": "kamu", "lu": "kamu", "elu": "kamu", "elo": "kamu", "ente": "kamu", "situ": "kamu",
    "hrs": "harus", "harus2": "harus", "kudu": "harus", "musti": "harus",
    "bs": "bisa", "bisa2": "bisa", "msh": "masih", "dlm": "dalam",
    "sblm": "sebelum", "stlh": "setelah", "pd": "pada", "pgn": "ingin", "pengen": "ingin",
    "liat": "lihat", "abis": "habis", "bkn": "bukan",
    "gpp": "tidak apa-apa", "gapapa": "tidak apa-apa",
    "cmn": "cuma", "cuman": "cuma", "mksh": "terima kasih", "makasih": "terima kasih",
    "moga": "semoga", "smoga": "semoga",
    "wkt": "waktu", "cpt": "cepat", "lg": "lagi", "lgi": "lagi",
    "sll": "selalu", "sllu": "selalu", "prnh": "pernah",
    "sndiri": "sendiri", "stiap": "setiap", "byk": "banyak", "dikit": "sedikit",
    "dtg": "datang", "krg": "kurang", "ato": "atau",
    "pke": "pakai", "pake": "pakai", "denger": "dengar", "kasih": "beri", "bikin": "buat",
    "brp": "berapa", "walopun": "walaupun", "walaupun": "walaupun", "meskipun": "meskipun", "meski": "meskipun",
    "kayanya": "sepertinya", "kykny": "sepertinya",
    "dpt": "dapat", "dapet": "dapat", "tggl": "tinggal", "tinggl": "tinggal",
    "tmn": "teman", "temen": "teman",
    "trnyata": "ternyata", "ternyta": "ternyata",
    "sbnrnya": "sebenarnya", "sebenernya": "sebenarnya", "sbnernya": "sebenarnya",
    "sbg": "sebagai", "sbgai": "sebagai", "trhdp": "terhadap", "thd": "terhadap", "thdp": "terhadap",
    "diantaranya": "di antaranya", "diantara": "di antara",
    "ngerti": "mengerti", "ngerasa": "merasa", "berasa": "terasa",
    "keliatan": "terlihat", "keliatannya": "terlihatnya",
    "nyari": "mencari", "nyoba": "mencoba", "nunggu": "menunggu",
    "ngasih": "memberi", "ngajak": "mengajak", "ngobrol": "berbicara",
    "nyadar": "sadar", "ngerasain": "merasakan", "ngebayangin": "membayangkan",
    "mikir": "berpikir", "mikirin": "memikirkan", "ngomongin": "membicarakan",
    "kesel": "kesal", "sebel": "sebal", "capek": "lelah", "cape": "lelah",
    "males": "malas", "mager": "malas gerak", "seneng": "senang",
    "parah": "sangat", "gila": "sangat", "anjir": "sangat", "anjay": "sangat",
    "mantap": "bagus", "mantul": "bagus", "keren": "bagus",
    "jelek": "buruk", "ancur": "hancur", "hancur": "hancur",
    "ngeri": "mengerikan", "serem": "menyeramkan",
    "bego": "bodoh", "goblok": "bodoh", "tolol": "bodoh", "bodo": "bodoh",
    "songong": "sombong", "belagu": "sombong",
    "curhat": "curahan hati", "japri": "pesan pribadi",
    "bener": "benar", "beneran": "benaran", "makanya": "makanya", "makannya": "makanya",
    "makin": "semakin", "kian": "semakin",
    "besok": "besok", "bsk": "besok", "kmrn": "kemarin", "kmarin": "kemarin", "kemaren": "kemarin",
    "td": "tadi", "entar": "nanti", "ntar": "nanti", "nti": "nanti",
    "pemrintah": "pemerintah", "pemerintahan": "pemerintah",
    "korup": "korupsi", "dikorupsi": "korupsi", "ngorupsi": "korupsi",
    "nyolong": "mencuri", "maling": "pencuri",
    "boong": "bohong", "bohong2": "bohong", "hoax": "hoaks", "hoak": "hoaks",
    "settingan": "rekayasa", "settingannya": "rekayasa", "php": "janji palsu",
    "ngamuk": "marah", "murka": "marah", "demo": "demonstrasi",
    "smpai": "sampai", "sampe": "sampai",
    "tuhh": "tuh", "sihh": "sih", "dehh": "deh",
    "pak": "bapak", "bu": "ibu", "min": "admin",
}

RE_URL = re.compile(r"(https?://\S+|www\.\S+)")
RE_MENTION = re.compile(r"@[A-Za-z0-9_]+")
RE_HASHTAG = re.compile(r"#([A-Za-z0-9_]+)")
RE_RT_PREFIX = re.compile(r"^\s*RT\s*@[A-Za-z0-9_]+\s*:\s*", flags=re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")
RE_MULTI_SPACE = re.compile(r"\s+")
RE_NON_ALNUM_PUNCT = re.compile(r"[^\w\s.,!?]")


def demojize_or_strip(text: str) -> str:
    if HAS_EMOJI_LIB:
        text = emoji.demojize(text, language="id" if "id" in emoji.LANGUAGES else "en")
        return text.replace("_", " ").replace(":", " ")
    return text


def fix_elongation(text: str) -> str:
    return RE_ELONGATION.sub(r"\1\1", text)


def normalize_slang(text: str) -> str:
    words = text.split()
    return " ".join(KAMUS_ALAY.get(w.lower().strip(".,!?"), w) for w in words)


def extract_hashtags(text: str):
    return RE_HASHTAG.findall(text)


def extract_mentions(text: str):
    return RE_MENTION.findall(text)


def build_raw_text(headline, mentions) -> str:
    """Gabungkan Headline + Mentions jadi satu teks mentah."""
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if not headline or headline.lower() == mentions.lower() or headline.lower() == "nan":
        return mentions or headline
    if not mentions:
        return headline
    return f"{headline}. {mentions}"


def clean_text(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return ""
    text = raw
    text = RE_RT_PREFIX.sub("", text)
    text = RE_URL.sub(" ", text)
    text = RE_MENTION.sub(" ", text)
    text = RE_HASHTAG.sub(r"\1", text)
    text = demojize_or_strip(text)
    text = RE_NON_ALNUM_PUNCT.sub(" ", text)
    text = fix_elongation(text)
    text = normalize_slang(text)
    text = RE_MULTI_SPACE.sub(" ", text).strip()
    return text


def make_dedup_key(text_clean: str) -> str:
    key = re.sub(r"[^\w\s]", "", text_clean.lower())
    key = RE_MULTI_SPACE.sub(" ", key).strip()
    return hashlib.md5(key.encode("utf-8")).hexdigest()

print("Fungsi cleaning siap.")


### 5b. Jalankan cleaning

In [ ]:
headline_col = df_raw[COL_HEADLINE] if COL_HEADLINE in df_raw.columns else pd.Series([""] * len(df_raw))
df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(headline_col, df_raw[COL_TEXT])]

tqdm.pandas(desc="Cleaning teks")
df_raw["hashtags"] = df_raw["text_raw_combined"].astype(str).apply(extract_hashtags)
df_raw["mentions_akun"] = df_raw["text_raw_combined"].astype(str).apply(extract_mentions)
df_raw["text_clean"] = df_raw["text_raw_combined"].astype(str).progress_apply(clean_text)

before = len(df_raw)
df_clean = df_raw[df_raw["text_clean"].str.split().str.len().fillna(0) >= 3].copy()
print(f"[INFO] Buang {before - len(df_clean)} baris teks kosong/terlalu pendek setelah cleaning")

df_clean["dedup_key"] = df_clean["text_clean"].apply(make_dedup_key)
before = len(df_clean)
df_clean["is_duplicate"] = df_clean.duplicated(subset="dedup_key", keep="first")
n_dup = df_clean["is_duplicate"].sum()
df_clean = df_clean[~df_clean["is_duplicate"]].drop(columns=["is_duplicate", "dedup_key"])
df_clean = df_clean.reset_index(drop=True)  # index rapi 0..n, dipakai di step-step berikutnya

print(f"[INFO] Buang {n_dup} duplikat/retweet identik")
print(f"[OK] {len(df_clean)} baris tersisa setelah cleaning\n")

print("Contoh sebelum vs sesudah cleaning:")
display(df_clean[["text_raw_combined", "text_clean"]].head(5))


## 6. Step 3 — Klasifikasi Tema (LLM)

Dikirim per-batch (default 15 post/panggilan) supaya hemat API call.

### 6a. Fungsi klasifikasi


In [ ]:
del df_master_clean

In [ ]:
# ---------------------------------------------------------------
# Step 3 - Klasifikasi Tema (hybrid: keyword dulu, sisanya baru LLM few-shot + multi-label)
# ---------------------------------------------------------------

TEST_MODE = False
N_TEST_ROWS = 100

# simpan salinan data bersih yang tidak akan pernah ketimpa lagi
# (df_clean di sini sudah pasti non-media, karena media sudah dibuang di Step 1b)
if "df_master_clean" not in globals():
    df_master_clean = df_clean.copy()
    print(f"[INFO] df_master_clean dibuat, {len(df_master_clean)} baris")
else:
    print(f"[INFO] df_master_clean sudah ada, {len(df_master_clean)} baris (tidak dibuat ulang)")

if TEST_MODE:
    df_run = df_master_clean.sample(n=min(N_TEST_ROWS, len(df_master_clean)), random_state=42).reset_index(drop=True)
    print(f"[TEST MODE] Memakai {len(df_run)} baris dari total {len(df_master_clean)} baris")
else:
    df_run = df_master_clean.copy()
    print(f"[FULL MODE] Memakai SEMUA {len(df_run)} baris")

texts_run = df_run["text_clean"].fillna("").tolist()
themes_run = [None] * len(texts_run)          # primary_theme (single, dipakai Step 5 ranking)
all_themes_run = [None] * len(texts_run)       # semua tema relevan (list, insight tambahan)
confidences_run = [0.0] * len(texts_run)

# tahap 1: keyword matching
for i, t in enumerate(texts_run):
    match = keyword_match_theme(t)
    if match:
        themes_run[i] = match
        all_themes_run[i] = [match]
        confidences_run[i] = 1.0  # keyword cocok persis, confidence dianggap tinggi

n_keyword = sum(1 for th in themes_run if th is not None)
need_llm_idx = [i for i, th in enumerate(themes_run) if th is None]
print(f"[INFO] {n_keyword} baris langsung kena keyword")
print(f"[INFO] {len(need_llm_idx)} baris dikirim ke LLM karena tidak ketemu keyword")

# tahap 2: sisanya baru dikirim ke LLM (few-shot + multi-label)
texts_for_llm = [texts_run[i] for i in need_llm_idx]

n_batches = (len(texts_for_llm) + BATCH_SIZE_CLASSIFY - 1) // BATCH_SIZE_CLASSIFY
for b in tqdm(range(n_batches), desc="Klasifikasi tema (LLM)"):
    start = b * BATCH_SIZE_CLASSIFY
    end = start + BATCH_SIZE_CLASSIFY
    batch = texts_for_llm[start:end]
    results = classify_theme_batch(batch)
    for r in results:
        local_idx = r["index"] - 1  # posisi DI DALAM batch ini saja (0..len(batch)-1)
        if 0 <= local_idx < len(batch):
            global_idx = need_llm_idx[start + local_idx]
            themes_raw = r.get("themes", ["lainnya"])
            themes_valid = [t for t in themes_raw if t in THEMES] or ["lainnya"]
            primary = r.get("primary_theme", themes_valid[0])
            themes_run[global_idx] = primary if primary in THEMES else themes_valid[0]
            all_themes_run[global_idx] = themes_valid
            confidences_run[global_idx] = r.get("confidence", 0.5)
    time.sleep(SLEEP_BETWEEN_CALLS)

# jaga-jaga kalau masih ada None (harusnya tidak terjadi)
themes_run = [th if th is not None else "lainnya" for th in themes_run]
all_themes_run = [at if at is not None else ["lainnya"] for at in all_themes_run]

df_run["theme"] = themes_run
df_run["all_themes"] = all_themes_run
df_run["theme_confidence"] = confidences_run
df_run["theme"] = df_run["theme"].fillna("lainnya")
df_run["theme_confidence"] = df_run["theme_confidence"].fillna(0.0)

print("\nDistribusi tema (primary):")
display(df_run["theme"].value_counts())

print("\nContoh teks dengan multi-tema (dari hasil LLM):")
display(df_run[df_run["all_themes"].apply(len) > 1][["text_clean", "all_themes", "theme"]].head(10))

display(df_run[["text_clean", "theme", "all_themes", "theme_confidence"]].head(10))

df_clean = df_run.copy()
print(f"\n[INFO] df_clean sekarang berisi {len(df_clean)} baris")

### 6b. Jalankan klasifikasi tema

In [ ]:
# ---------------------------------------------------------------
# TEST: ambil sampel dari SEMUA tema sekaligus (karhutla, polusi_udara, erupsi, + lainnya)
# ---------------------------------------------------------------
import re as _re

N_PER_THEME = 5

sample_parts = []
for theme, desc in THEME_DESCRIPTIONS.items():
    if theme == "lainnya":
        continue
    # pecah deskripsi per koma, bersihkan, escape biar aman jadi regex
    kw_list = [_re.escape(k.strip()) for k in desc.split(",") if k.strip()]
    pattern = "|".join(kw_list)

    match = df_clean[df_clean["text_clean"].str.contains(pattern, case=False, na=False, regex=True)]
    if not match.empty:
        n = min(N_PER_THEME, len(match))
        sample_parts.append(match.sample(n, random_state=42))
        print(f"[INFO] Tema '{theme}': ditemukan {len(match)} kandidat, ambil {n}")
    else:
        print(f"[WARN] Tema '{theme}': tidak ada kandidat yang cocok")

# kontrol "lainnya"
sample_parts.append(df_clean.sample(min(N_PER_THEME, len(df_clean)), random_state=99))

df_test = pd.concat(sample_parts).drop_duplicates(subset="text_clean").reset_index(drop=True)
test_texts = df_test["text_clean"].tolist()

print(f"\n[TEST] Total {len(df_test)} teks akan dites (multi-tema)")
display(df_test[["text_clean"]])

# kalau jumlahnya lebih dari BATCH_SIZE_CLASSIFY, tetap dipecah per batch spy tidak kepotong
n_batches = (len(test_texts) + BATCH_SIZE_CLASSIFY - 1) // BATCH_SIZE_CLASSIFY
all_results = []
for b in range(n_batches):
    start = b * BATCH_SIZE_CLASSIFY
    end = start + BATCH_SIZE_CLASSIFY
    batch = test_texts[start:end]
    results = classify_theme_batch(batch)
    for r in results:
        r["index"] += start  # geser index biar cocok sama posisi global di test_texts
    all_results.extend(results)
    time.sleep(SLEEP_BETWEEN_CALLS)

print("\nHasil klasifikasi per tema:")
for r in sorted(all_results, key=lambda x: x["index"]):
    idx = r["index"] - 1
    primary = r.get("primary_theme", "?")
    all_th = r.get("themes", [])
    conf = r.get("confidence", "?")
    print(f"[{primary:15s}] all_themes={all_th} (conf={conf}) -> {test_texts[idx][:90]}")

## 7. Step 4 — Filter Akun Media & NER




### 7b. NER — ekstraksi entitas (lokasi, instansi, tokoh)

Hanya dijalankan untuk baris **non-media** & tema **relevan** (bukan "lainnya")
supaya hemat biaya API. Hapus filter ini di cell bawah kalau butuh NER untuk semua baris.


In [ ]:
FEWSHOT_NER_EXAMPLES = """
Contoh ekstraksi (belajar dari pola ini):

Teks: "BMKG mencatat titik api meningkat di Kalimantan Tengah, Gubernur Sugianto turun langsung ke lokasi"
Jawaban: {"lokasi": ["Kalimantan Tengah"], "instansi": ["BMKG"], "tokoh": ["Sugianto"]}

Teks: "PVMBG menaikkan status Gunung Semeru menjadi Awas, BPBD Lumajang siapkan jalur evakuasi"
Jawaban: {"lokasi": ["Gunung Semeru", "Lumajang"], "instansi": ["PVMBG", "BPBD"], "tokoh": []}

Teks: "Kualitas udara memburuk"
Jawaban: {"lokasi": [], "instansi": [], "tokoh": []}
"""


def build_ner_prompt(batch_texts):
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    return f"""Ekstrak entitas penting dari tiap teks berbahasa Indonesia berikut.
Kategori entitas:
- lokasi: nama tempat/daerah/gunung (misal "Kalimantan Tengah", "Gunung Semeru")
- instansi: nama lembaga/organisasi (misal "BMKG", "BNPB", "KLHK", "Pemprov DKI")
- tokoh: nama orang yang disebut (pejabat, tokoh publik, dll)

{FEWSHOT_NER_EXAMPLES}

Sekarang ekstrak entitas dari teks berikut. Kalau tidak ada entitas di kategori
tertentu, gunakan array kosong (lihat contoh ke-3 di atas) — JANGAN mengarang entitas.

Teks:
{numbered}

Jawab HANYA dengan JSON array, tanpa penjelasan, tanpa markdown code block:
[
  {{"index": 1, "lokasi": ["..."], "instansi": ["..."], "tokoh": ["..."]}},
  {{"index": 2, "lokasi": [], "instansi": [], "tokoh": []}}
]
Jumlah item HARUS sama persis dengan jumlah teks ({len(batch_texts)} item)."""


def extract_ner_batch(batch_texts, retry=6):
    prompt = build_ner_prompt(batch_texts)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if len(parsed) != len(batch_texts):
                raise ValueError(f"Jumlah hasil ({len(parsed)}) != jumlah input ({len(batch_texts)})")
            return parsed, True
        except Exception as e:
            is_overload = "503" in str(e) or "UNAVAILABLE" in str(e)
            wait = 20 * (attempt + 1) if is_overload else 5 * (attempt + 1)
            print(f"[WARN] NER batch gagal (percobaan {attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    print(f"[ERROR] Batch NER ini gagal total setelah {retry} percobaan")
    return [{"index": i + 1, "lokasi": [], "instansi": [], "tokoh": []} for i in range(len(batch_texts))], False


target_idx = df_clean[df_clean["theme"] != "lainnya"].index.tolist()
print(f"Menjalankan NER untuk {len(target_idx)} baris (tema relevan) ...")

texts_all = df_clean["text_clean"].fillna("").tolist()
lokasi_col = [[] for _ in range(len(df_clean))]
instansi_col = [[] for _ in range(len(df_clean))]
tokoh_col = [[] for _ in range(len(df_clean))]
ner_failed_col = [False for _ in range(len(df_clean))]

n_batches = (len(target_idx) + BATCH_SIZE_CLASSIFY - 1) // BATCH_SIZE_CLASSIFY
for b in tqdm(range(n_batches), desc="NER ekstraksi entitas"):
    idx_batch = target_idx[b * BATCH_SIZE_CLASSIFY:(b + 1) * BATCH_SIZE_CLASSIFY]
    text_batch = [texts_all[i] for i in idx_batch]
    results, success = extract_ner_batch(text_batch)
    for r, df_idx in zip(results, idx_batch):
        lokasi_col[df_idx] = r.get("lokasi", [])
        instansi_col[df_idx] = r.get("instansi", [])
        tokoh_col[df_idx] = r.get("tokoh", [])
        ner_failed_col[df_idx] = not success
    time.sleep(SLEEP_BETWEEN_CALLS)

df_clean["entities_lokasi"] = lokasi_col
df_clean["entities_instansi"] = instansi_col
df_clean["entities_tokoh"] = tokoh_col
df_clean["ner_failed"] = ner_failed_col

n_failed = df_clean.loc[target_idx, "ner_failed"].sum()
if n_failed > 0:
    print(f"[WARN] {n_failed} baris gagal diproses NER setelah semua percobaan, entitasnya kosong bukan berarti tidak ada")

print("\nContoh hasil NER:")
display(df_clean.loc[target_idx, ["text_clean", "entities_lokasi", "entities_instansi", "entities_tokoh", "ner_failed"]].head(10))

## 8. Step 5 — Ranking Top Author per Tema

Hanya akun **non-media**. Skor = kombinasi jumlah post + engagement + followers
(bobot diatur di bagian Konfigurasi).


In [ ]:
df_topic = df_clean[(~df_clean["is_media"]) & (df_clean["theme"] != "lainnya")].copy()
print(f"[INFO] {len(df_topic)} baris tersisa setelah filter non-media & tema relevan")

for col in [COL_LIKES, COL_RETWEETS]:
    if col in df_topic.columns:
        df_topic[col] = pd.to_numeric(df_topic[col], errors="coerce").fillna(0)
    else:
        df_topic[col] = 0

if COL_FOLLOWERS in df_topic.columns:
    df_topic[COL_FOLLOWERS] = pd.to_numeric(df_topic[COL_FOLLOWERS], errors="coerce").fillna(0)
else:
    df_topic[COL_FOLLOWERS] = 0

df_topic["engagement"] = df_topic[COL_LIKES] + df_topic[COL_RETWEETS]

top_authors_per_theme = {}

for theme in THEMES:
    if theme == "lainnya":
        continue
    sub = df_topic[df_topic["theme"] == theme]
    if sub.empty:
        print(f"[INFO] Tidak ada data non-media untuk tema '{theme}'")
        continue

    agg = (
        sub.groupby(COL_AUTHOR_ID)
        .agg(
            jumlah_post=(COL_AUTHOR_ID, "count"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"),
        )
        .reset_index()
        .rename(columns={COL_AUTHOR_ID: "author_id"})
    )

    max_post = agg["jumlah_post"].max() or 1
    max_eng = agg["total_engagement"].max() or 1
    max_followers = agg["followers"].max() or 1
    agg["score"] = (
        WEIGHT_POST_COUNT * (agg["jumlah_post"] / max_post)
        + WEIGHT_ENGAGEMENT * (agg["total_engagement"] / max_eng)
        + WEIGHT_FOLLOWERS * (agg["followers"] / max_followers)
    )

    agg = agg.sort_values("score", ascending=False).head(TOP_N_AUTHORS_PER_THEME)
    agg.insert(0, "theme", theme)
    agg["rank"] = range(1, len(agg) + 1)
    top_authors_per_theme[theme] = agg

    print(f"\n=== Top Author: {theme} ===")
    display(agg[["rank", "author_id", "jumlah_post", "total_engagement", "followers", "score"]])

df_top_authors = (
    pd.concat(top_authors_per_theme.values(), ignore_index=True)
    if top_authors_per_theme else pd.DataFrame(columns=["theme", "author_id", "jumlah_post", "total_engagement", "followers", "score", "rank"])
)


## 9. Step 6 — Ringkasan & Sentimen per Top Author (LLM)

Untuk tiap top author, semua post-nya (tentang tema itu) dikirim ke LLM
untuk diringkas + ditentukan sentimen keseluruhannya. Sentimen per-post yang
sudah ada dari tool monitoring (`Sentiment`) ikut dikirim sebagai konteks
tambahan, bukan patokan mutlak.

### 9a. Fungsi summarization


In [ ]:
SENTIMENT_OPTIONS = ["positif", "negatif", "kontroversial"]


def build_summary_prompt(author_label, theme, texts, tool_sentiment_note=""):
    joined = "\n".join(f"- {t}" for t in texts)
    context_note = ""
    if tool_sentiment_note:
        context_note = (
            f"\nSebagai referensi tambahan (bukan patokan mutlak), tool monitoring sosial media "
            f"sebelumnya sudah memberi label sentimen per-post untuk akun ini dengan distribusi: "
            f"{tool_sentiment_note}. Gunakan ini hanya sebagai bahan pertimbangan, "
            f"keputusan akhir tetap berdasarkan isi teks yang kamu baca sendiri.\n"
        )
    return f"""Berikut kumpulan cuitan dari akun "{author_label}" tentang topik "{theme}":

{joined}
{context_note}
Tugas kamu:
1. Ringkas dalam 2-3 kalimat apa pandangan/narasi utama yang disampaikan akun ini soal topik tersebut.
2. Tentukan sentimen KESELURUHAN akun ini terhadap topik, pilih SATU dari: {", ".join(SENTIMENT_OPTIONS)}.
   - "kontroversial" dipakai jika pendapat akun ini memicu perdebatan/pro-kontra atau menyampaikan klaim yang kontroversial.
   - "netral" dipakai jika akun hanya menyampaikan informasi tanpa opini/emosi yang jelas.
3. Berikan alasan singkat (1 kalimat) untuk sentimen tersebut.

Jawab HANYA dengan JSON, tanpa penjelasan tambahan, tanpa markdown code block:
{{"summary": "...", "sentiment": "...", "reason": "..."}}"""


def summarize_author(author_label, theme, texts, tool_sentiment_note="", retry=3):
    prompt = build_summary_prompt(author_label, theme, texts, tool_sentiment_note)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if parsed.get("sentiment") not in SENTIMENT_OPTIONS:
                parsed["sentiment"] = "netral"
            return parsed
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"[WARN] gagal summarize {author_label} (percobaan {attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return {"summary": "(gagal diringkas otomatis)", "sentiment": "netral", "reason": "error API"}

print("Fungsi summarization siap.")

### 9b. Jalankan summarization untuk semua top author

In [ ]:
pd.set_option("display.max_colwidth", None)   # jangan potong isi kolom teks panjang
pd.set_option("display.max_rows", None)       # tampilkan semua baris (opsional, hati-hati kalau datanya banyak)

results = []

for _, row in df_top_authors.iterrows():
    author_id = row["author_id"]
    theme = row["theme"]

    author_posts_df = df_topic[(df_topic[COL_AUTHOR_ID] == author_id) & (df_topic["theme"] == theme)]
    posts = author_posts_df["text_clean"].dropna().tolist()

    tool_sentiment_note = ""
    if COL_SENTIMENT_TOOL in author_posts_df.columns:
        counts = author_posts_df[COL_SENTIMENT_TOOL].dropna().value_counts()
        if not counts.empty:
            total = counts.sum()
            tool_sentiment_note = ", ".join(f"{label} {round(100 * n / total)}%" for label, n in counts.items())

    posts = posts[:MAX_POSTS_PER_AUTHOR_SUMMARY]
    if not posts:
        continue

    print(f"Meringkas @{author_id} | tema={theme} | {len(posts)} post ...")
    result = summarize_author(author_id, theme, posts, tool_sentiment_note)

    results.append({
        "theme": theme,
        "rank": int(row["rank"]),
        "author_id": author_id,
        "score": round(row["score"], 2),
        "summary": result.get("summary", ""),
        "sentiment": result.get("sentiment", "netral"),
        "reason": result.get("reason", ""),
    })

df_summary = pd.DataFrame(results)

print("\n=== Hasil akhir: ringkasan & sentimen per top author ===")
display(df_summary)

# --- Export ke Excel ---
file_name = "hasil_analisis_top_author.xlsx"
df_summary.to_excel(file_name, index=False)
print(f"\n[OK] Data berhasil diekspor ke {file_name}")

from google.colab import files
files.download(file_name)